In [1]:
#Importación de librerias generales
import os
import json
import pandas as pd
import pickle

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef

#Imprimir los graficos en el notebook
%matplotlib inline

In [2]:
#Variables generales
ruta_model = "../Model/"
ruta_df = "../Datasets/"
ruta_metrics = "../Metrics/"
ruta_metrics_test = "../Metrics/TEST/"
semilla = 111

## Prueba Modelos


### Evaluación de rendimiento - TRAIN

In [3]:
def load_evaluation_results(directory):
    results = []
    filenames = []
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            filepath = os.path.join(directory, filename)
            with open(filepath, 'r') as file:
                result = json.load(file)
                results.append(result)
                filenames.append(filename)
    return results, filenames



In [4]:
def process_results_to_dataframe(results, filenames):
    data = []
    
    for result, filename in zip(results, filenames):
        row = {
            'filename': filename,
            'accuracy': result['accuracy'],
            'precision': result['precision'],
            'recall': result['recall'],
            'f1_score': result['f1_score'],
            'roc_auc': result['roc_auc'],
            'balanced_accuracy': result['balanced_accuracy'],
            'cohen_kappa': result['cohen_kappa'],
            'matthews_corrcoef': result['matthews_corrcoef']
        }
        data.append(row)
    
    df = pd.DataFrame(data)
    return df



In [5]:
def highlight_top2(s):
    is_max = s == s.max()
    is_second = s == s.nlargest(2).iloc[-1]
    return ['background-color: lightgreen' if v else 'background-color: yellow' if w else '' for v, w in zip(is_max, is_second)]

def display_results_table(df):
    styled_df = df.style.apply(highlight_top2, subset=['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'balanced_accuracy', 'cohen_kappa', 'matthews_corrcoef'])
    display(styled_df)

In [6]:
# Load the results from JSON files
results, filenames = load_evaluation_results(ruta_metrics)
df_results = process_results_to_dataframe(results, filenames)

# Display
display_results_table(df_results)

,filename,accuracy,precision,recall,f1_score,roc_auc,balanced_accuracy,cohen_kappa,matthews_corrcoef
0,G_IM_A.json,0.404106,0.876772,0.404106,0.496865,0.588825,0.588825,0.046381,0.106528
1,G_IM_S.json,0.580511,0.684014,0.580511,0.460207,0.529577,0.529577,0.065102,0.153118
2,G_LR_A.json,0.580240,0.670904,0.580240,0.458597,0.526550,0.526550,0.058726,0.139782
3,G_LR_S.json,0.583229,0.696678,0.583229,0.464370,0.532437,0.532437,0.071381,0.165942
4,KNN_IM_A.json,0.885227,0.854320,0.885227,0.867666,0.536438,0.536438,0.093867,0.100241
5,KNN_IM_S.json,0.994932,0.994977,0.994932,0.994929,0.994298,0.994298,0.989725,0.989777
6,KNN_LR_A.json,0.999261,0.999261,0.999261,0.999260,0.999173,0.999173,0.998500,0.998501
7,KNN_LR_S.json,0.998202,0.998207,0.998202,0.998201,0.997986,0.997986,0.996357,0.996363
8,RF_IM_A.json,0.908354,0.849842,0.908354,0.868300,0.503230,0.503230,0.011370,0.030085
9,RF_IM_S.json,0.994870,0.994915,0.994870,0.994867,0.994243,0.994243,0.989601,0.989652


### Test 

### 1. Replicar la lógica basada en la modificación de los datos en el dataset test

In [7]:
def get_columns_from_csv(file_path):
    df = pd.read_csv(file_path, nrows=0)
    return df.columns.tolist()


* Interpolation Method

In [8]:
#  SMOTE - retornar columnas a usar
column_names_IM_S = get_columns_from_csv('{}Interpolation_Method_SMOTE_Train_Modificado.csv'.format(ruta_df))
column_names_IM_S

['Mean', 'Sum', 'Norm', 'Percentile_85', 'Percentile_65', 'FLAG']

In [9]:
# ADASYN - retornar columnas a usar
column_names_IM_A = get_columns_from_csv('{}Interpolation_Method_ADASYN_Train_Modificado.csv'.format(ruta_df))
column_names_IM_A

['Mean', 'Sum', 'Percentile_85', 'Norm', 'Median_Absolute_Deviation', 'FLAG']

In [10]:
"""
procedemos a replicar la logica empleada en el pre procesamiento de los datos.
"""

#  Carga de dataset TEST
df_IM = pd.read_csv('{}Interpolation_Method_Test.csv'.format(ruta_df))


In [11]:
"""En este contexto, el dataset modificado deriva en la limitacion de las columnas"""

'En este contexto, el dataset modificado deriva en la limitacion de las columnas'

In [12]:
# implementar filtros según logica - SMOTE

df_IM_S = df_IM[column_names_IM_S]
df_IM_S.sample(3)

,Mean,Sum,Norm,Percentile_85,Percentile_65,FLAG
10386,8.627880,8921.228296,289.784897,11.4335,9.04,0
8310,1.672853,1729.730254,103.048607,5.3900,0.00,0
9667,2.801664,2896.920991,177.636749,8.9935,0.00,0


In [13]:
# implementar filtros según logica - ADASYN 

df_IM_A = df_IM[column_names_IM_A]
df_IM_A.sample(3)

,Mean,Sum,Percentile_85,Norm,Median_Absolute_Deviation,FLAG
10857,3.407719,3523.580989,6.1520,134.735603,1.34,1
11254,1.482207,1532.602158,2.5505,62.067170,0.34,0
143,11.520890,11912.599801,18.1220,412.980545,2.01,0


* Linear Regression

In [14]:
#  SMOTE - retornar columnas a usar
column_names_LR_S = get_columns_from_csv('{}Linear_Regression_SMOTE_Train_Modificado.csv'.format(ruta_df))
column_names_LR_S

['Mean',
 'Sum',
 'Median',
 'Norm',
 'Median_Absolute_Deviation',
 'Standard_Dev',
 'IQR',
 'FLAG']

In [15]:
# ADASYN - retornar columnas a usar
column_names_LR_A = get_columns_from_csv('{}Linear_Regression_ADASYN_Train_Modificado.csv'.format(ruta_df))
column_names_LR_A

['Sum', 'Mean', 'Median', 'Norm', 'Median_Absolute_Deviation', 'FLAG']

In [16]:
"""
procedemos a replicar la logica empleada en el pre procesamiento de los datos.
"""

#  Carga de dataset TEST
df_LR = pd.read_csv('{}Interpolation_Method_Test.csv'.format(ruta_df))


In [17]:
"""En este contexto, el dataset modificado deriva en la limitacion de las columnas"""

'En este contexto, el dataset modificado deriva en la limitacion de las columnas'

In [18]:
# implementar filtros según logica - SMOTE

df_LR_S = df_IM[column_names_LR_S]
df_LR_S.sample(3)

,Mean,Sum,Median,Norm,Median_Absolute_Deviation,Standard_Dev,IQR,FLAG
11517,2.326308,2405.402818,2.68,97.525998,1.855,1.946927,3.900,0
9318,8.528086,8818.041400,8.10,286.658839,1.450,2.597987,3.255,0
3192,0.000000,0.000000,0.00,0.000000,0.000,0.000000,0.000,0


In [19]:
# implementar filtros según logica - ADASYN 

df_LR_A = df_IM[column_names_LR_A]
df_LR_A.sample(3)

,Sum,Mean,Median,Norm,Median_Absolute_Deviation,FLAG
4095,2275.452511,2.200631,0.650,110.933655,0.520,0
3667,1878.277581,1.816516,1.595,79.424037,1.480,1
5352,7907.978792,7.647949,6.650,292.946856,2.845,0


### 2. importar los modelos a evaluar

In [20]:
#!pip install joblib
import joblib

In [21]:
def load_model_and_evaluate(model_path, df, output_metrics_path, target_column='FLAG'):
    # Cargar el modelo
    model = joblib.load(model_path)
    
    # Separar características y la columna objetivo
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    # Hacer predicciones
    y_pred = model.predict(X)
    
    # Calcular métricas de evaluación
    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average='weighted'),
        'recall': recall_score(y, y_pred, average='weighted'),
        'f1_score': f1_score(y, y_pred, average='weighted'),
        'roc_auc': roc_auc_score(y, y_pred, multi_class='ovr'),
        'confusion_matrix': confusion_matrix(y, y_pred).tolist(),
        'classification_report': classification_report(y, y_pred, output_dict=True),
        'balanced_accuracy': balanced_accuracy_score(y, y_pred),
        'cohen_kappa': cohen_kappa_score(y, y_pred),
        'matthews_corrcoef': matthews_corrcoef(y, y_pred)
    }
    
    # Guardar las métricas en un archivo JSON
    with open(output_metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)
    
    return metrics

* Interpolation Method

In [22]:
# SMOTE - df_IM_S 

In [23]:

load_model_and_evaluate(model_path = '{}RF_IM_S.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}RF_IM_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.7869729389553178,
 'precision': 0.8613849425460897,
 'recall': 0.7869729389553178,
 'f1_score': 0.8186021997994518,
 'roc_auc': 0.5940859287158881,
 'confusion_matrix': [[9595, 1978], [730, 409]],
 'classification_report': {'0': {'precision': 0.9292978208232445,
   'recall': 0.8290849390823468,
   'f1-score': 0.8763357384236002,
   'support': 11573},
  '1': {'precision': 0.17134478424801006,
   'recall': 0.35908691834942935,
   'f1-score': 0.2319909245604084,
   'support': 1139},
  'accuracy': 0.7869729389553178,
  'macro avg': {'precision': 0.5503213025356273,
   'recall': 0.5940859287158881,
   'f1-score': 0.5541633314920044,
   'support': 12712},
  'weighted avg': {'precision': 0.8613849425460897,
   'recall': 0.7869729389553178,
   'f1-score': 0.8186021997994518,
   'support': 12712}},
 'balanced_accuracy': 0.5940859287158881,
 'cohen_kappa': 0.1259575808267357,
 'matthews_corrcoef': 0.1376157910017258}

In [24]:
load_model_and_evaluate(model_path = '{}RL_IM_S.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}RL_IM_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.4080396475770925,
 'precision': 0.8853597409892886,
 'recall': 0.4080396475770925,
 'f1_score': 0.49956973978407265,
 'roc_auc': 0.6072118681375704,
 'confusion_matrix': [[4219, 7354], [171, 968]],
 'classification_report': {'0': {'precision': 0.9610478359908884,
   'recall': 0.36455543074397306,
   'f1-score': 0.5285973814445907,
   'support': 11573},
  '1': {'precision': 0.11631819274212929,
   'recall': 0.8498683055311677,
   'f1-score': 0.20462953176197018,
   'support': 1139},
  'accuracy': 0.4080396475770925,
  'macro avg': {'precision': 0.5386830143665089,
   'recall': 0.6072118681375703,
   'f1-score': 0.36661345660328043,
   'support': 12712},
  'weighted avg': {'precision': 0.8853597409892886,
   'recall': 0.4080396475770925,
   'f1-score': 0.49956973978407265,
   'support': 12712}},
 'balanced_accuracy': 0.6072118681375703,
 'cohen_kappa': 0.05579780918733224,
 'matthews_corrcoef': 0.12879873035749825}

In [25]:
load_model_and_evaluate(model_path = '{}KNN_IM_S.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}KNN_IM_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.7011485210824417,
 'precision': 0.8604602208347509,
 'recall': 0.7011485210824417,
 'f1_score': 0.7616406328385742,
 'roc_auc': 0.5908816629666991,
 'confusion_matrix': [[8393, 3180], [619, 520]],
 'classification_report': {'0': {'precision': 0.9313138038171327,
   'recall': 0.7252225006480602,
   'f1-score': 0.8154481418508622,
   'support': 11573},
  '1': {'precision': 0.14054054054054055,
   'recall': 0.456540825285338,
   'f1-score': 0.21492043810704695,
   'support': 1139},
  'accuracy': 0.7011485210824417,
  'macro avg': {'precision': 0.5359271721788366,
   'recall': 0.5908816629666991,
   'f1-score': 0.5151842899789546,
   'support': 12712},
  'weighted avg': {'precision': 0.8604602208347509,
   'recall': 0.7011485210824417,
   'f1-score': 0.7616406328385742,
   'support': 12712}},
 'balanced_accuracy': 0.5908816629666991,
 'cohen_kappa': 0.09026845516212867,
 'matthews_corrcoef': 0.11428247727982796}

In [26]:
load_model_and_evaluate(model_path = '{}G_IM_S.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}G_IM_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.906859660163625,
 'precision': 0.8694760474431265,
 'recall': 0.906859660163625,
 'f1_score': 0.8772013669442845,
 'roc_auc': 0.5328842442829792,
 'confusion_matrix': [[11440, 133], [1051, 88]],
 'classification_report': {'0': {'precision': 0.9158594187815227,
   'recall': 0.9885077335176704,
   'f1-score': 0.9507978723404256,
   'support': 11573},
  '1': {'precision': 0.39819004524886875,
   'recall': 0.07726075504828797,
   'f1-score': 0.12941176470588237,
   'support': 1139},
  'accuracy': 0.906859660163625,
  'macro avg': {'precision': 0.6570247320151957,
   'recall': 0.5328842442829792,
   'f1-score': 0.540104818523154,
   'support': 12712},
  'weighted avg': {'precision': 0.8694760474431265,
   'recall': 0.906859660163625,
   'f1-score': 0.8772013669442845,
   'support': 12712}},
 'balanced_accuracy': 0.5328842442829792,
 'cohen_kappa': 0.10329974318556434,
 'matthews_corrcoef': 0.14371693909984365}

In [27]:
load_model_and_evaluate(model_path = '{}XGB_IM_S.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}XGB_IM_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.7685651353052234,
 'precision': 0.8639533364437235,
 'recall': 0.7685651353052234,
 'f1_score': 0.807597063134282,
 'roc_auc': 0.6053481784180688,
 'confusion_matrix': [[9307, 2266], [676, 463]],
 'classification_report': {'0': {'precision': 0.9322848843033157,
   'recall': 0.8041994297070768,
   'f1-score': 0.86351827797365,
   'support': 11573},
  '1': {'precision': 0.16965921582997434,
   'recall': 0.4064969271290606,
   'f1-score': 0.23940020682523266,
   'support': 1139},
  'accuracy': 0.7685651353052234,
  'macro avg': {'precision': 0.5509720500666451,
   'recall': 0.6053481784180688,
   'f1-score': 0.5514592423994413,
   'support': 12712},
  'weighted avg': {'precision': 0.8639533364437235,
   'recall': 0.7685651353052234,
   'f1-score': 0.807597063134282,
   'support': 12712}},
 'balanced_accuracy': 0.6053481784180688,
 'cohen_kappa': 0.12931819014645518,
 'matthews_corrcoef': 0.14655801069550103}

In [28]:
# ADASYN - df_IM_A

In [29]:

load_model_and_evaluate(model_path = '{}RF_IM_A.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}RF_IM_A.json'.format(ruta_metrics)
                        , target_column = 'FLAG')

C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- Percentile_65
Feature names seen at fit time, yet now missing:
- Median_Absolute_Deviation

  warnings.warn(message, FutureWarning)


{'accuracy': 0.9083543108873505,
 'precision': 0.8498424884106633,
 'recall': 0.9083543108873505,
 'f1_score': 0.8683001033260801,
 'roc_auc': 0.5032302488452316,
 'confusion_matrix': [[11536, 37], [1128, 11]],
 'classification_report': {'0': {'precision': 0.9109286165508528,
   'recall': 0.9968029033094271,
   'f1-score': 0.9519329950076331,
   'support': 11573},
  '1': {'precision': 0.22916666666666666,
   'recall': 0.009657594381035996,
   'f1-score': 0.018534119629317607,
   'support': 1139},
  'accuracy': 0.9083543108873505,
  'macro avg': {'precision': 0.5700476416087598,
   'recall': 0.5032302488452316,
   'f1-score': 0.48523355731847534,
   'support': 12712},
  'weighted avg': {'precision': 0.8498424884106633,
   'recall': 0.9083543108873505,
   'f1-score': 0.8683001033260801,
   'support': 12712}},
 'balanced_accuracy': 0.5032302488452316,
 'cohen_kappa': 0.011369978237359746,
 'matthews_corrcoef': 0.030084634843580205}

In [30]:
load_model_and_evaluate(model_path = '{}RL_IM_A.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}RL_IM_A.json'.format(ruta_metrics)
                        , target_column = 'FLAG')

C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- Percentile_65
Feature names seen at fit time, yet now missing:
- Median_Absolute_Deviation

  warnings.warn(message, FutureWarning)


{'accuracy': 0.20154185022026433,
 'precision': 0.8996442828747504,
 'recall': 0.20154185022026433,
 'f1_score': 0.2188977923142908,
 'roc_auc': 0.5488144235693764,
 'confusion_matrix': [[1455, 10118], [32, 1107]],
 'classification_report': {'0': {'precision': 0.9784801613987895,
   'recall': 0.1257236671563121,
   'f1-score': 0.22281776416539054,
   'support': 11573},
  '1': {'precision': 0.09861915367483297,
   'recall': 0.9719051799824407,
   'f1-score': 0.17906826269815596,
   'support': 1139},
  'accuracy': 0.20154185022026433,
  'macro avg': {'precision': 0.5385496575368112,
   'recall': 0.5488144235693764,
   'f1-score': 0.20094301343177323,
   'support': 12712},
  'weighted avg': {'precision': 0.8996442828747504,
   'recall': 0.20154185022026433,
   'f1-score': 0.2188977923142908,
   'support': 12712}},
 'balanced_accuracy': 0.5488144235693764,
 'cohen_kappa': 0.01955779569363436,
 'matthews_corrcoef': 0.08675896060825782}

In [31]:
load_model_and_evaluate(model_path = '{}KNN_IM_A.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}KNN_IM_A.json'.format(ruta_metrics)
                        , target_column = 'FLAG')

C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- Percentile_65
Feature names seen at fit time, yet now missing:
- Median_Absolute_Deviation

  warnings.warn(message, FutureWarning)


{'accuracy': 0.8852265575833858,
 'precision': 0.8543199589482549,
 'recall': 0.8852265575833858,
 'f1_score': 0.8676661001746564,
 'roc_auc': 0.5364384663009107,
 'confusion_matrix': [[11126, 447], [1012, 127]],
 'classification_report': {'0': {'precision': 0.9166254737188994,
   'recall': 0.961375615657133,
   'f1-score': 0.9384673780102063,
   'support': 11573},
  '1': {'precision': 0.22125435540069685,
   'recall': 0.11150131694468832,
   'f1-score': 0.14827787507297138,
   'support': 1139},
  'accuracy': 0.8852265575833858,
  'macro avg': {'precision': 0.5689399145597981,
   'recall': 0.5364384663009106,
   'f1-score': 0.5433726265415888,
   'support': 12712},
  'weighted avg': {'precision': 0.8543199589482549,
   'recall': 0.8852265575833858,
   'f1-score': 0.8676661001746564,
   'support': 12712}},
 'balanced_accuracy': 0.5364384663009106,
 'cohen_kappa': 0.09386691983480244,
 'matthews_corrcoef': 0.10024100465328273}

In [32]:
load_model_and_evaluate(model_path = '{}G_IM_A.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}G_IM_A.json'.format(ruta_metrics)
                        , target_column = 'FLAG')

C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- Percentile_65
Feature names seen at fit time, yet now missing:
- Median_Absolute_Deviation

  warnings.warn(message, FutureWarning)


{'accuracy': 0.4041063561988672,
 'precision': 0.8767723917914504,
 'recall': 0.4041063561988672,
 'f1_score': 0.49686545777220464,
 'roc_auc': 0.5888247879798328,
 'confusion_matrix': [[4210, 7363], [212, 927]],
 'classification_report': {'0': {'precision': 0.9520578923563998,
   'recall': 0.36377775857599587,
   'f1-score': 0.5264145045326665,
   'support': 11573},
  '1': {'precision': 0.11182147165259348,
   'recall': 0.8138718173836699,
   'f1-score': 0.1966274260260897,
   'support': 1139},
  'accuracy': 0.4041063561988672,
  'macro avg': {'precision': 0.5319396820044966,
   'recall': 0.5888247879798328,
   'f1-score': 0.3615209652793781,
   'support': 12712},
  'weighted avg': {'precision': 0.8767723917914504,
   'recall': 0.4041063561988672,
   'f1-score': 0.49686545777220464,
   'support': 12712}},
 'balanced_accuracy': 0.5888247879798328,
 'cohen_kappa': 0.04638121530218786,
 'matthews_corrcoef': 0.1065276580460248}

In [33]:
load_model_and_evaluate(model_path = '{}XGB_IM_A.pkl'.format(ruta_model)
                        , df = df_IM_S
                        , output_metrics_path = '{}XGB_IM_A.json'.format(ruta_metrics)
                        , target_column = 'FLAG')

{'accuracy': 0.9107142857142857,
 'precision': 0.9186887337560142,
 'recall': 0.9107142857142857,
 'f1_score': 0.8684706916577564,
 'roc_auc': 0.5017559262510974,
 'confusion_matrix': [[11573, 0], [1135, 4]],
 'classification_report': {'0': {'precision': 0.9106861819326408,
   'recall': 1.0,
   'f1-score': 0.9532556319756188,
   'support': 11573},
  '1': {'precision': 1.0,
   'recall': 0.003511852502194908,
   'f1-score': 0.00699912510936133,
   'support': 1139},
  'accuracy': 0.9107142857142857,
  'macro avg': {'precision': 0.9553430909663204,
   'recall': 0.5017559262510974,
   'f1-score': 0.48012737854249005,
   'support': 12712},
  'weighted avg': {'precision': 0.9186887337560142,
   'recall': 0.9107142857142857,
   'f1-score': 0.8684706916577564,
   'support': 12712}},
 'balanced_accuracy': 0.5017559262510974,
 'cohen_kappa': 0.00637599940058009,
 'matthews_corrcoef': 0.05655259098162057}

* Linear Regression

In [34]:
# SMOTE - df_LR_S 

In [35]:

load_model_and_evaluate(model_path = '{}RF_LR_S.pkl'.format(ruta_model)
                        , df = df_LR_S
                        , output_metrics_path = '{}RF_LR_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.8160006293266205,
 'precision': 0.8524085963741649,
 'recall': 0.8160006293266205,
 'f1_score': 0.8328375717181021,
 'roc_auc': 0.5558066833378257,
 'confusion_matrix': [[10101, 1472], [867, 272]],
 'classification_report': {'0': {'precision': 0.9209518599562363,
   'recall': 0.8728073965263976,
   'f1-score': 0.8962335300119781,
   'support': 11573},
  '1': {'precision': 0.1559633027522936,
   'recall': 0.23880597014925373,
   'f1-score': 0.18869233437391608,
   'support': 1139},
  'accuracy': 0.8160006293266205,
  'macro avg': {'precision': 0.5384575813542649,
   'recall': 0.5558066833378257,
   'f1-score': 0.5424629321929471,
   'support': 12712},
  'weighted avg': {'precision': 0.8524085963741649,
   'recall': 0.8160006293266205,
   'f1-score': 0.8328375717181021,
   'support': 12712}},
 'balanced_accuracy': 0.5558066833378257,
 'cohen_kappa': 0.09005105419706727,
 'matthews_corrcoef': 0.09265398134081738}

In [36]:
load_model_and_evaluate(model_path = '{}RL_LR_S.pkl'.format(ruta_model)
                        , df = df_LR_S
                        , output_metrics_path = '{}RL_LR_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.35973882945248586,
 'precision': 0.88254073065041,
 'recall': 0.35973882945248586,
 'f1_score': 0.44408691055034527,
 'roc_auc': 0.5870170472627585,
 'confusion_matrix': [[3589, 7984], [155, 984]],
 'classification_report': {'0': {'precision': 0.9586004273504274,
   'recall': 0.3101183789855699,
   'f1-score': 0.46862962721159496,
   'support': 11573},
  '1': {'precision': 0.10972346119536129,
   'recall': 0.8639157155399473,
   'f1-score': 0.19471653309587417,
   'support': 1139},
  'accuracy': 0.35973882945248586,
  'macro avg': {'precision': 0.5341619442728943,
   'recall': 0.5870170472627586,
   'f1-score': 0.33167308015373453,
   'support': 12712},
  'weighted avg': {'precision': 0.88254073065041,
   'recall': 0.35973882945248586,
   'f1-score': 0.44408691055034527,
   'support': 12712}},
 'balanced_accuracy': 0.5870170472627586,
 'cohen_kappa': 0.04246243535927641,
 'matthews_corrcoef': 0.109044422496195}

In [37]:
load_model_and_evaluate(model_path = '{}KNN_LR_S.pkl'.format(ruta_model)
                        , df = df_LR_S
                        , output_metrics_path = '{}KNN_LR_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.7043738200125865,
 'precision': 0.8532121109806209,
 'recall': 0.7043738200125865,
 'f1_score': 0.7628969155691168,
 'roc_auc': 0.5633654883945838,
 'confusion_matrix': [[8508, 3065], [693, 446]],
 'classification_report': {'0': {'precision': 0.924682099771764,
   'recall': 0.7351594227944354,
   'f1-score': 0.8191007990757678,
   'support': 11573},
  '1': {'precision': 0.12702933637140415,
   'recall': 0.3915715539947322,
   'f1-score': 0.1918279569892473,
   'support': 1139},
  'accuracy': 0.7043738200125865,
  'macro avg': {'precision': 0.525855718071584,
   'recall': 0.5633654883945838,
   'f1-score': 0.5054643780325075,
   'support': 12712},
  'weighted avg': {'precision': 0.8532121109806209,
   'recall': 0.7043738200125865,
   'f1-score': 0.7628969155691168,
   'support': 12712}},
 'balanced_accuracy': 0.5633654883945838,
 'cohen_kappa': 0.06536620555398887,
 'matthews_corrcoef': 0.08095332490759323}

In [38]:
load_model_and_evaluate(model_path = '{}G_LR_S.pkl'.format(ruta_model)
                        , df = df_LR_S
                        , output_metrics_path = '{}G_LR_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.907960981749528,
 'precision': 0.8727061783827861,
 'recall': 0.907960981749528,
 'f1_score': 0.8785435325282535,
 'roc_auc': 0.535863765734282,
 'confusion_matrix': [[11448, 125], [1045, 94]],
 'classification_report': {'0': {'precision': 0.9163531577683502,
   'recall': 0.9891989976669835,
   'f1-score': 0.9513836948391923,
   'support': 11573},
  '1': {'precision': 0.4292237442922374,
   'recall': 0.08252853380158033,
   'f1-score': 0.13843888070692192,
   'support': 1139},
  'accuracy': 0.907960981749528,
  'macro avg': {'precision': 0.6727884510302938,
   'recall': 0.535863765734282,
   'f1-score': 0.5449112877730571,
   'support': 12712},
  'weighted avg': {'precision': 0.8727061783827861,
   'recall': 0.907960981749528,
   'f1-score': 0.8785435325282535,
   'support': 12712}},
 'balanced_accuracy': 0.535863765734282,
 'cohen_kappa': 0.11279959560997743,
 'matthews_corrcoef': 0.1574400778625303}

In [39]:
load_model_and_evaluate(model_path = '{}XGB_LR_S.pkl'.format(ruta_model)
                        , df = df_LR_S
                        , output_metrics_path = '{}XGB_LR_S.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.802391441157961,
 'precision': 0.8527941573415517,
 'recall': 0.802391441157961,
 'f1_score': 0.8251399514125795,
 'roc_auc': 0.5590183836663202,
 'confusion_matrix': [[9901, 1672], [840, 299]],
 'classification_report': {'0': {'precision': 0.9217949911553859,
   'recall': 0.8555257927935712,
   'f1-score': 0.887424935018374,
   'support': 11573},
  '1': {'precision': 0.15169964485032977,
   'recall': 0.26251097453906935,
   'f1-score': 0.1922829581993569,
   'support': 1139},
  'accuracy': 0.802391441157961,
  'macro avg': {'precision': 0.5367473180028578,
   'recall': 0.5590183836663203,
   'f1-score': 0.5398539466088654,
   'support': 12712},
  'weighted avg': {'precision': 0.8527941573415517,
   'recall': 0.802391441157961,
   'f1-score': 0.8251399514125795,
   'support': 12712}},
 'balanced_accuracy': 0.5590183836663203,
 'cohen_kappa': 0.08879705739995636,
 'matthews_corrcoef': 0.09314005180588947}

In [40]:
# ADASYN - df_LR_A

In [41]:

load_model_and_evaluate(model_path = '{}RF_LR_A.pkl'.format(ruta_model)
                        , df = df_LR_A
                        , output_metrics_path = '{}RF_LR_A.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.7830396475770925,
 'precision': 0.8513538857773997,
 'recall': 0.7830396475770925,
 'f1_score': 0.8131489864920172,
 'roc_auc': 0.5551184157791511,
 'confusion_matrix': [[9638, 1935], [823, 316]],
 'classification_report': {'0': {'precision': 0.9213268329987573,
   'recall': 0.8328004838849046,
   'f1-score': 0.874829808477807,
   'support': 11573},
  '1': {'precision': 0.14038205242114615,
   'recall': 0.2774363476733977,
   'f1-score': 0.1864306784660767,
   'support': 1139},
  'accuracy': 0.7830396475770925,
  'macro avg': {'precision': 0.5308544427099517,
   'recall': 0.5551184157791511,
   'f1-score': 0.5306302434719419,
   'support': 12712},
  'weighted avg': {'precision': 0.8513538857773997,
   'recall': 0.7830396475770925,
   'f1-score': 0.8131489864920172,
   'support': 12712}},
 'balanced_accuracy': 0.5551184157791511,
 'cohen_kappa': 0.0765477928062922,
 'matthews_corrcoef': 0.08247782737005425}

In [42]:
load_model_and_evaluate(model_path = '{}RL_LR_A.pkl'.format(ruta_model)
                        , df = df_LR_A
                        , output_metrics_path = '{}RL_LR_A.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.14694776589049716,
 'precision': 0.9055147719343645,
 'recall': 0.14694776589049716,
 'f1_score': 0.12476110760019525,
 'roc_auc': 0.5271421697152109,
 'confusion_matrix': [[740, 10833], [11, 1128]],
 'classification_report': {'0': {'precision': 0.9853528628495339,
   'recall': 0.0639419338114577,
   'f1-score': 0.12009087958455048,
   'support': 11573},
  '1': {'precision': 0.09430649611236519,
   'recall': 0.990342405618964,
   'f1-score': 0.17221374045801527,
   'support': 1139},
  'accuracy': 0.14694776589049716,
  'macro avg': {'precision': 0.5398296794809496,
   'recall': 0.5271421697152109,
   'f1-score': 0.14615231002128287,
   'support': 12712},
  'weighted avg': {'precision': 0.9055147719343645,
   'recall': 0.14694776589049716,
   'f1-score': 0.12476110760019525,
   'support': 12712}},
 'balanced_accuracy': 0.5271421697152109,
 'cohen_kappa': 0.010275083058920886,
 'matthews_corrcoef': 0.06575907299147045}

In [43]:
load_model_and_evaluate(model_path = '{}KNN_LR_A.pkl'.format(ruta_model)
                        , df = df_LR_A
                        , output_metrics_path = '{}KNN_LR_A.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.6920232850849591,
 'precision': 0.8552804711746702,
 'recall': 0.6920232850849591,
 'f1_score': 0.754619690869358,
 'roc_auc': 0.570830450853372,
 'confusion_matrix': [[8315, 3258], [657, 482]],
 'classification_report': {'0': {'precision': 0.9267721801159162,
   'recall': 0.7184826751922578,
   'f1-score': 0.8094426867851059,
   'support': 11573},
  '1': {'precision': 0.12887700534759358,
   'recall': 0.4231782265144864,
   'f1-score': 0.19758147161303546,
   'support': 1139},
  'accuracy': 0.6920232850849591,
  'macro avg': {'precision': 0.5278245927317549,
   'recall': 0.5708304508533721,
   'f1-score': 0.5035120791990707,
   'support': 12712},
  'weighted avg': {'precision': 0.8552804711746702,
   'recall': 0.6920232850849591,
   'f1-score': 0.754619690869358,
   'support': 12712}},
 'balanced_accuracy': 0.5708304508533721,
 'cohen_kappa': 0.06980372818068104,
 'matthews_corrcoef': 0.0887880273010198}

In [44]:
load_model_and_evaluate(model_path = '{}G_LR_A.pkl'.format(ruta_model)
                        , df = df_LR_A
                        , output_metrics_path = '{}G_LR_A.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.9073316551290119,
 'precision': 0.8713020130134765,
 'recall': 0.9073316551290119,
 'f1_score': 0.8781534344957107,
 'roc_auc': 0.5355181336596254,
 'confusion_matrix': [[11440, 133], [1045, 94]],
 'classification_report': {'0': {'precision': 0.9162995594713657,
   'recall': 0.9885077335176704,
   'f1-score': 0.9510349987530136,
   'support': 11573},
  '1': {'precision': 0.41409691629955947,
   'recall': 0.08252853380158033,
   'f1-score': 0.1376281112737921,
   'support': 1139},
  'accuracy': 0.9073316551290119,
  'macro avg': {'precision': 0.6651982378854626,
   'recall': 0.5355181336596254,
   'f1-score': 0.5443315550134028,
   'support': 12712},
  'weighted avg': {'precision': 0.8713020130134765,
   'recall': 0.9073316551290119,
   'f1-score': 0.8781534344957107,
   'support': 12712}},
 'balanced_accuracy': 0.5355181336596254,
 'cohen_kappa': 0.11115901802797157,
 'matthews_corrcoef': 0.1531996487404648}

In [45]:
load_model_and_evaluate(model_path = '{}XGB_LR_A.pkl'.format(ruta_model)
                        , df = df_LR_A
                        , output_metrics_path = '{}XGB_LR_A.json'.format(ruta_metrics_test)
                        , target_column = 'FLAG')

{'accuracy': 0.7899622404027691,
 'precision': 0.8520542722834888,
 'recall': 0.7899622404027691,
 'f1_score': 0.817532748034096,
 'roc_auc': 0.5573372583866036,
 'confusion_matrix': [[9730, 1843], [827, 312]],
 'classification_report': {'0': {'precision': 0.9216633513308705,
   'recall': 0.8407500216020046,
   'f1-score': 0.8793492995933121,
   'support': 11573},
  '1': {'precision': 0.14477958236658933,
   'recall': 0.2739244951712028,
   'f1-score': 0.1894353369763206,
   'support': 1139},
  'accuracy': 0.7899622404027691,
  'macro avg': {'precision': 0.53322146684873,
   'recall': 0.5573372583866038,
   'f1-score': 0.5343923182848164,
   'support': 12712},
  'weighted avg': {'precision': 0.8520542722834888,
   'recall': 0.7899622404027691,
   'f1-score': 0.817532748034096,
   'support': 12712}},
 'balanced_accuracy': 0.5573372583866038,
 'cohen_kappa': 0.08178710460634953,
 'matthews_corrcoef': 0.08728866658822593}

### Evaluación de rendimiento - TEST

In [46]:
# Load the results from JSON files
results, filenames = load_evaluation_results(ruta_metrics_test)
df_results = process_results_to_dataframe(results, filenames)

# Display
display_results_table(df_results)

,filename,accuracy,precision,recall,f1_score,roc_auc,balanced_accuracy,cohen_kappa,matthews_corrcoef
0,G_IM_S.json,0.906860,0.869476,0.906860,0.877201,0.532884,0.532884,0.103300,0.143717
1,G_LR_A.json,0.907332,0.871302,0.907332,0.878153,0.535518,0.535518,0.111159,0.153200
2,G_LR_S.json,0.907961,0.872706,0.907961,0.878544,0.535864,0.535864,0.112800,0.157440
3,KNN_IM_S.json,0.701149,0.860460,0.701149,0.761641,0.590882,0.590882,0.090268,0.114282
4,KNN_LR_A.json,0.692023,0.855280,0.692023,0.754620,0.570830,0.570830,0.069804,0.088788
5,KNN_LR_S.json,0.704374,0.853212,0.704374,0.762897,0.563365,0.563365,0.065366,0.080953
6,RF_IM_S.json,0.786973,0.861385,0.786973,0.818602,0.594086,0.594086,0.125958,0.137616
7,RF_LR_A.json,0.783040,0.851354,0.783040,0.813149,0.555118,0.555118,0.076548,0.082478
8,RF_LR_S.json,0.816001,0.852409,0.816001,0.832838,0.555807,0.555807,0.090051,0.092654
9,RL_IM_S.json,0.408040,0.885360,0.408040,0.499570,0.607212,0.607212,0.055798,0.128799
